In [ ]:
# Train the model with Random forest regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
model=RandomForestRegressor(n_estimators=500,max_depth=20,min_samples_split=2,min_samples_leaf=1,max_features='sqrt',random_state=42, n_jobs=-1)
model.fit(X_train,Y_train)
Y_pred=model.predict(X_test)
print(Y_pred[0:10])

feature_importance=pd.DataFrame({
  'Feature':X_train.columns,
  "Importance":model.feature_importances_
})

feature_importance=feature_importance.sort_values(by='Importance',ascending=False)
print(feature_importance)

#prediction on test data
Y_pred=model.predict(X_test)

mean_abs_err=mean_absolute_error(Y_test,Y_pred)
mean_squ_err=mean_squared_error(Y_test,Y_pred)
rmae=np.sqrt(mean_squ_err)



print("mean absolute error:",mean_abs_err)
print("mean square error:",mean_squ_err)
print("root mean square error:",rmae)


# FUTURE FORECASTING (NEXT 30 DAYS)

future_days = 30
future_predictions = []

# Get last available data for each Store & Product
last_data = df.sort_values("Date").groupby(["Store ID", "Product ID"]).tail(1)

current_data = last_data.copy()

for i in range(future_days):
    
    # Move to next day
    current_data["Date"] = current_data["Date"] + pd.Timedelta(days=1)
    
    # Update time-based features
    current_data["Month"] = current_data["Date"].dt.month
    current_data["Is_Weekend"] = current_data["Date"].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)
    
    # IMPORTANT: Keep other features constant (simplification)
    # You can improve later if needed
    
    # Predict
    preds = model.predict(current_data[input_features])
    
    current_data["Units Sold"] = preds
    
    # Update lag feature (VERY IMPORTANT)
    current_data["Sales_lag_1"] = current_data["Units Sold"]
    
    # Store prediction
    future_predictions.append(current_data.copy())

# Combine all future predictions
future_df = pd.concat(future_predictions)

print(future_df.head())

#visualization on predictions
plt.figure(figsize=(12,6))
plt.plot(test["Date"][:100],Y_test[:100].reset_index(drop=True),label="Actual")
plt.plot(test["Date"][:100],Y_pred[:100],label="Predicted")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.title("Sales Forcast vs Actual")
plt.show()


plt.figure(figsize=(12,6))

# Last historical data
plt.plot(df["Date"].tail(100), df["Units Sold"].tail(100), label="Historical")

# Future forecast
plt.plot(future_df["Date"], future_df["Units Sold"], label="Forecast")

plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.title("Future Sales Forecast (Next 30 Days)")
plt.legend()
plt.xticks(rotation=45)
plt.show()


